## Model Comparison for Structural Phase Classification

This notebook compares seven classifiers using the iteration 6
structural phase dataset: XGBoost, Ridge, SVC, Gaussian Process,
Random Forest, Gradient Boosting, and TabPFN.

### Workflow

1. Aggregate records by compound, retaining the first feature values
   and the most frequent label.
2. Create a stratified training/test split (80/20).
3. Select the top 15 features using XGBoost on the training set.
4. Evaluate model configurations using five-fold stratified
   cross-validation with balanced accuracy as the selection metric.
5. Export test predictions and comparison metrics, ranked by
   test balanced accuracy.

Precision, recall, and F1 are macro-averaged. Feature selection occurs
before cross-validation. Predictions use each model's predict method.
TabPFN runs on CPU using the specified local checkpoint.
Run the code cells sequentially.

In [1]:
# ============================================================
# 1. Import packages
# ============================================================

import os
import numpy as np
import pandas as pd

from xgboost import XGBClassifier
from tabpfn import TabPFNClassifier

from sklearn.ensemble import (
    GradientBoostingClassifier,
    RandomForestClassifier
)
from sklearn.gaussian_process import (
    GaussianProcessClassifier
)
from sklearn.gaussian_process.kernels import (
    ConstantKernel,
    RBF
)
from sklearn.linear_model import RidgeClassifier
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import (
    LabelEncoder,
    StandardScaler
)
from sklearn.svm import SVC

D:\app\miniconda\envs\mp311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# ============================================================
# 2. Basic settings
# ============================================================

DATA_FILE = (
    "data/compound_6_Structural_label.csv"
)

TARGET_COL = "label"
GROUP_COL = "compound"

N_SELECTED_FEATURES = 15
TEST_SIZE = 0.20
RANDOM_STATE = 42

TABPFN_MODEL_PATH = (
    "tabpfn_models/tabpfn-v2-classifier.ckpt"
)

OUTPUT_DIR = "model_comparison/label"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


In [3]:
# ============================================================
# 3. Data preparation
# ============================================================

data = pd.read_csv(DATA_FILE)

data = data.drop(
    columns=[
        col for col in data.columns
        if str(col).startswith("Unnamed:")
    ],
    errors="ignore"
)

feature_cols = [
    col for col in data.columns
    if col not in [GROUP_COL, TARGET_COL]
]


def most_frequent_label(values):

    return values.mode().iloc[0]


aggregation_rules = {
    col: "first"
    for col in feature_cols
}

aggregation_rules[TARGET_COL] = (
    most_frequent_label
)

data = (
    data
    .groupby(GROUP_COL, as_index=False)
    .agg(aggregation_rules)
)

label_encoder = LabelEncoder()

data["target_encoded"] = (
    label_encoder.fit_transform(
        data[TARGET_COL]
    )
)

train_data, test_data = train_test_split(
    data,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=data["target_encoded"]
)

train_data = train_data.reset_index(drop=True)
test_data = test_data.reset_index(drop=True)

X_train_all = train_data[feature_cols]
X_test_all = test_data[feature_cols]

y_train = (
    train_data["target_encoded"]
    .to_numpy()
)

y_test = (
    test_data["target_encoded"]
    .to_numpy()
)

print("Total samples:", len(data))
print("Training samples:", len(train_data))
print("Test samples:", len(test_data))
print("Candidate features:", len(feature_cols))
print("Classes:", label_encoder.classes_)

print("\nTraining class counts:")
print(
    train_data["target_encoded"]
    .value_counts()
    .sort_index()
)

print("\nTest class counts:")
print(
    test_data["target_encoded"]
    .value_counts()
    .sort_index()
)

Total samples: 156
Training samples: 124
Test samples: 32
Candidate features: 25
Classes: [0 1]

Training class counts:
target_encoded
0    81
1    43
Name: count, dtype: int64

Test class counts:
target_encoded
0    21
1    11
Name: count, dtype: int64


In [4]:
# ============================================================
# 4. XGBoost feature selection
# ============================================================

feature_selector = XGBClassifier(
    n_estimators=100,
    max_depth=2,
    learning_rate=0.05,
    importance_type="gain",
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

feature_selector.fit(
    X_train_all,
    y_train
)

feature_importance = pd.Series(
    feature_selector.feature_importances_,
    index=feature_cols
).sort_values(
    ascending=False
)

selected_features = (
    feature_importance
    .head(N_SELECTED_FEATURES)
    .index
    .tolist()
)

X_train = train_data[selected_features]
X_test = test_data[selected_features]

print("\nSelected features:")

for index, feature in enumerate(
    selected_features,
    start=1
):
    print(f"{index}. {feature}")



Selected features:
1. AB_AN
2. AB_SIE
3. AB_FIE
4. AB_EN
5. C_FIE
6. C_EA
7. AB_EA
8. AB_Nv
9. C_CR
10. AB_D
11. AB_AW
12. AB_Nd
13. AB_AR
14. AB_CR
15. AB_TC


In [5]:
# ============================================================
# 5. Cross-validation
# ============================================================

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)


# ============================================================
# 6. Model training and evaluation function
# ============================================================

def train_and_evaluate(
    model_name,
    model,
    param_grid,
    n_jobs=-1
):

    search = GridSearchCV(
        estimator=model,
        param_grid=param_grid,
        scoring="balanced_accuracy",
        cv=cv,
        n_jobs=n_jobs,
        refit=True
    )

    search.fit(
        X_train,
        y_train
    )

    best_model = search.best_estimator_

    train_prediction = best_model.predict(
        X_train
    )

    test_prediction = best_model.predict(
        X_test
    )

    train_accuracy = accuracy_score(
        y_train,
        train_prediction
    )

    train_balanced_accuracy = (
        balanced_accuracy_score(
            y_train,
            train_prediction
        )
    )

    train_precision = precision_score(
        y_train,
        train_prediction,
        average="macro",
        zero_division=0
    )

    train_recall = recall_score(
        y_train,
        train_prediction,
        average="macro",
        zero_division=0
    )

    train_f1 = f1_score(
        y_train,
        train_prediction,
        average="macro",
        zero_division=0
    )

    test_accuracy = accuracy_score(
        y_test,
        test_prediction
    )

    test_balanced_accuracy = (
        balanced_accuracy_score(
            y_test,
            test_prediction
        )
    )

    test_precision = precision_score(
        y_test,
        test_prediction,
        average="macro",
        zero_division=0
    )

    test_recall = recall_score(
        y_test,
        test_prediction,
        average="macro",
        zero_division=0
    )

    test_f1 = f1_score(
        y_test,
        test_prediction,
        average="macro",
        zero_division=0
    )

    print(f"\n===== {model_name} =====")
    print("Best parameters:", search.best_params_)

    print(
        f"Train: Accuracy="
        f"{train_accuracy:.6f} | "
        f"Balanced Accuracy="
        f"{train_balanced_accuracy:.6f} | "
        f"Precision={train_precision:.6f} | "
        f"Recall={train_recall:.6f} | "
        f"F1={train_f1:.6f}"
    )

    print(
        f"Test: Accuracy="
        f"{test_accuracy:.6f} | "
        f"Balanced Accuracy="
        f"{test_balanced_accuracy:.6f} | "
        f"Precision={test_precision:.6f} | "
        f"Recall={test_recall:.6f} | "
        f"F1={test_f1:.6f}"
    )

    result = {
        "Model": model_name,
        "Train_Accuracy": train_accuracy,
        "Train_Balanced_Accuracy": (
            train_balanced_accuracy
        ),
        "Train_Macro_Precision": (
            train_precision
        ),
        "Train_Macro_Recall": train_recall,
        "Train_Macro_F1": train_f1,
        "Test_Accuracy": test_accuracy,
        "Test_Balanced_Accuracy": (
            test_balanced_accuracy
        ),
        "Test_Macro_Precision": (
            test_precision
        ),
        "Test_Macro_Recall": test_recall,
        "Test_Macro_F1": test_f1,
        "Best_parameters": str(
            search.best_params_
        )
    }

    prediction_result = pd.DataFrame({
        GROUP_COL: test_data[GROUP_COL],
        "y_true": label_encoder.inverse_transform(
            y_test
        ),
        "y_pred": label_encoder.inverse_transform(
            test_prediction.astype(int)
        )
    })

    prediction_result.to_csv(
        f"{OUTPUT_DIR}/{model_name}_test_predictions.csv",
        index=False
    )

    return best_model, result

In [6]:
# ============================================================
# 7. XGBoost
# ============================================================

xgb_model = XGBClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbosity=0
)

xgb_grid = {
    "n_estimators": [
        50,
        100,
        200
    ],
    "max_depth": [
        2,
        3,
        4
    ],
    "learning_rate": [
        0.03,
        0.05,
        0.10
    ]
}

best_xgb, xgb_result = train_and_evaluate(
    model_name="XGBoost",
    model=xgb_model,
    param_grid=xgb_grid
)


===== XGBoost =====
Best parameters: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}
Train: Accuracy=0.991935 | Balanced Accuracy=0.993827 | Precision=0.988636 | Recall=0.993827 | F1=0.991147
Test: Accuracy=0.906250 | Balanced Accuracy=0.885281 | Precision=0.904545 | Recall=0.885281 | F1=0.893688


In [7]:
# ============================================================
# 8. Ridge classifier
# ============================================================

ridge_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        RidgeClassifier()
    )
])

ridge_grid = {
    "model__alpha": [
        0.001,
        0.01,
        0.1,
        1,
        10,
        100
    ],
    "model__class_weight": [
        None,
        "balanced"
    ]
}

best_ridge, ridge_result = train_and_evaluate(
    model_name="Ridge Classifier",
    model=ridge_model,
    param_grid=ridge_grid
)


===== Ridge Classifier =====
Best parameters: {'model__alpha': 10, 'model__class_weight': 'balanced'}
Train: Accuracy=0.653226 | Balanced Accuracy=0.674562 | Precision=0.658333 | Recall=0.674562 | F1=0.646583
Test: Accuracy=0.656250 | Balanced Accuracy=0.629870 | Precision=0.625000 | Recall=0.629870 | F1=0.626723


In [8]:
# ============================================================
# 9. Support vector classifier
# ============================================================

svc_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        SVC()
    )
])

svc_grid = {
    "model__C": [
        0.1,
        1,
        10,
        100
    ],
    "model__kernel": [
        "linear",
        "rbf"
    ],
    "model__gamma": [
        "scale",
        0.01,
        0.10,
        1.0
    ],
    "model__class_weight": [
        None,
        "balanced"
    ]
}

best_svc, svc_result = train_and_evaluate(
    model_name="SVC",
    model=svc_model,
    param_grid=svc_grid
)


===== SVC =====
Best parameters: {'model__C': 100, 'model__class_weight': 'balanced', 'model__gamma': 1.0, 'model__kernel': 'rbf'}
Train: Accuracy=0.951613 | Balanced Accuracy=0.962963 | Precision=0.938776 | Recall=0.962963 | F1=0.948161
Test: Accuracy=0.781250 | Balanced Accuracy=0.768398 | Precision=0.758333 | Recall=0.768398 | F1=0.762460


In [9]:
# ============================================================
# 10. Gaussian process classifier
# ============================================================

gpc_kernel = (
    ConstantKernel(
        constant_value=1.0,
        constant_value_bounds=(1e-2, 1e2)
    )
    * RBF(
        length_scale=1.0,
        length_scale_bounds=(1e-2, 1e2)
    )
)

gpc_model = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "model",
        GaussianProcessClassifier(
            kernel=gpc_kernel,
            n_restarts_optimizer=5,
            random_state=RANDOM_STATE
        )
    )
])

gpc_grid = {}

best_gpc, gpc_result = train_and_evaluate(
    model_name="Gaussian Process",
    model=gpc_model,
    param_grid=gpc_grid
)



===== Gaussian Process =====
Best parameters: {}
Train: Accuracy=0.991935 | Balanced Accuracy=0.993827 | Precision=0.988636 | Recall=0.993827 | F1=0.991147
Test: Accuracy=0.812500 | Balanced Accuracy=0.792208 | Precision=0.792208 | Recall=0.792208 | F1=0.792208


In [10]:
# ============================================================
# 11. Random forest
# ============================================================

rf_model = RandomForestClassifier(
    random_state=RANDOM_STATE,
    n_jobs=1
)

rf_grid = {
    "n_estimators": [
        100,
        300,
        400
    ],
    "max_depth": [
        3,
        5,
        8
    ],
    "min_samples_leaf": [
        2,
        4,
        6
    ],
    "class_weight": [
        None,
        "balanced"
    ]
}

best_rf, rf_result = train_and_evaluate(
    model_name="Random Forest",
    model=rf_model,
    param_grid=rf_grid
)


===== Random Forest =====
Best parameters: {'class_weight': 'balanced', 'max_depth': 8, 'min_samples_leaf': 2, 'n_estimators': 100}
Train: Accuracy=0.967742 | Balanced Accuracy=0.975309 | Precision=0.957447 | Recall=0.975309 | F1=0.965120
Test: Accuracy=0.843750 | Balanced Accuracy=0.837662 | Precision=0.825000 | Recall=0.837662 | F1=0.830329


In [11]:
# ============================================================
# 12. Gradient boosting
# ============================================================

gbdt_model = GradientBoostingClassifier(
    random_state=RANDOM_STATE
)

gbdt_grid = {
    "n_estimators": [
        50,
        100,
        150
    ],
    "learning_rate": [
        0.01,
        0.03,
        0.05
    ],
    "max_depth": [
        1,
        2,
        3
    ],
    "min_samples_leaf": [
        3,
        5,
        8
    ],
    "subsample": [
        0.7,
        0.85,
        1.0
    ]
}

best_gbdt, gbdt_result = (
    train_and_evaluate(
        model_name="Gradient Boosting",
        model=gbdt_model,
        param_grid=gbdt_grid
    )
)


===== Gradient Boosting =====
Best parameters: {'learning_rate': 0.05, 'max_depth': 3, 'min_samples_leaf': 3, 'n_estimators': 150, 'subsample': 1.0}
Train: Accuracy=0.991935 | Balanced Accuracy=0.988372 | Precision=0.993902 | Recall=0.988372 | F1=0.991050
Test: Accuracy=0.875000 | Balanced Accuracy=0.861472 | Precision=0.861472 | Recall=0.861472 | F1=0.861472


In [12]:
# ============================================================
# 13. TabPFN
# ============================================================

tabpfn_model = TabPFNClassifier(
    device="cpu",
    model_path=TABPFN_MODEL_PATH,
    random_state=RANDOM_STATE
)

tabpfn_grid = {}

best_tabpfn, tabpfn_result = (
    train_and_evaluate(
        model_name="TabPFN",
        model=tabpfn_model,
        param_grid=tabpfn_grid,
        n_jobs=1
    )
)



===== TabPFN =====
Best parameters: {}
Train: Accuracy=0.943548 | Balanced Accuracy=0.918605 | Precision=0.960227 | Recall=0.918605 | F1=0.934986
Test: Accuracy=0.781250 | Balanced Accuracy=0.703463 | Precision=0.801282 | Recall=0.703463 | F1=0.719650


In [13]:
# ============================================================
# 14. Model comparison
# ============================================================

comparison_results = pd.DataFrame([
    xgb_result,
    ridge_result,
    svc_result,
    gpc_result,
    rf_result,
    gbdt_result,
    tabpfn_result
])

comparison_results = (
    comparison_results
    .sort_values(
        "Test_Balanced_Accuracy",
        ascending=False
    )
    .reset_index(drop=True)
)

print("\nFinal model comparison:")

print(
    comparison_results[
        [
            "Model",
            "Train_Accuracy",
            "Train_Balanced_Accuracy",
            "Train_Macro_F1",
            "Test_Accuracy",
            "Test_Balanced_Accuracy",
            "Test_Macro_F1"
        ]
    ]
)

comparison_results.to_csv(
    f"{OUTPUT_DIR}/model_comparison_results.csv",
    index=False
)


Final model comparison:
               Model  Train_Accuracy  Train_Balanced_Accuracy  Train_Macro_F1  \
0            XGBoost        0.991935                 0.993827        0.991147   
1  Gradient Boosting        0.991935                 0.988372        0.991050   
2      Random Forest        0.967742                 0.975309        0.965120   
3   Gaussian Process        0.991935                 0.993827        0.991147   
4                SVC        0.951613                 0.962963        0.948161   
5             TabPFN        0.943548                 0.918605        0.934986   
6   Ridge Classifier        0.653226                 0.674562        0.646583   

   Test_Accuracy  Test_Balanced_Accuracy  Test_Macro_F1  
0        0.90625                0.885281       0.893688  
1        0.87500                0.861472       0.861472  
2        0.84375                0.837662       0.830329  
3        0.81250                0.792208       0.792208  
4        0.78125                0.76